# Question 1:

In [1]:
import numpy as np

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline

from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler

from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans

from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

from sklearn.metrics import confusion_matrix

In [2]:
merged_df = pd.read_csv("merged-df.csv")

# Initial Data Cleaning

In [3]:
merged_df.head()

,Unnamed: 0,Country/Territory,Code,Year,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS,Total Chronic Deaths,GNI_per_capita,HDI,LifeExpectancy
0,0,Afghanistan,AFG,1990,1116,371,44899,11580,2108,3709,5945,2673,34,72435,2684.550019,0.273,45.9672
1,1,Afghanistan,AFG,1991,1136,374,45492,11796,2120,3724,6050,2728,41,73461,2276.289409,0.279,46.6631
2,2,Afghanistan,AFG,1992,1162,378,46557,12218,2153,3776,6223,2830,48,75345,2059.868084,0.287,47.5955
3,3,Afghanistan,AFG,1993,1187,384,47951,12634,2195,3862,6445,2943,56,77657,1525.533426,0.297,51.4664
4,4,Afghanistan,AFG,1994,1211,391,49308,12914,2231,3932,6664,3027,63,79741,1087.961890,0.292,51.4945


First, we perform some initial data preparation, to further clean up the data from `merged-df.csv`.

In [4]:
merged_df.drop("Unnamed: 0", axis = 1, inplace = True)

We drop ANY entries with an N/A value, just in case this interferes with our classification/clustering models. In any case, each entry is a separate "snapshot" that the model will predict for, so removing entries just means that we will have less data to train on, not necessarily that the model will get biased towards a certain prediction.

In [5]:
len(merged_df)

5520

In [6]:
merged_df = merged_df.drop(merged_df[merged_df.isna().any(axis=1)].index)
len(merged_df)

4905

In [7]:
merged_df.head()

,Country/Territory,Code,Year,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS,Total Chronic Deaths,GNI_per_capita,HDI,LifeExpectancy
0,Afghanistan,AFG,1990,1116,371,44899,11580,2108,3709,5945,2673,34,72435,2684.550019,0.273,45.9672
1,Afghanistan,AFG,1991,1136,374,45492,11796,2120,3724,6050,2728,41,73461,2276.289409,0.279,46.6631
2,Afghanistan,AFG,1992,1162,378,46557,12218,2153,3776,6223,2830,48,75345,2059.868084,0.287,47.5955
3,Afghanistan,AFG,1993,1187,384,47951,12634,2195,3862,6445,2943,56,77657,1525.533426,0.297,51.4664
4,Afghanistan,AFG,1994,1211,391,49308,12914,2231,3932,6664,3027,63,79741,1087.961890,0.292,51.4945


# Exploratory Data Analysis

Generating Classification Metric #1 (based on HDI)

In [8]:
merged_df["HDI"].describe()

count    4905.000000
mean        0.668152
std         0.165012
min         0.216000
25%         0.542000
50%         0.694000
75%         0.796000
max         0.962000
Name: HDI, dtype: float64

In [9]:
merged_df["HDI_category"] = merged_df.apply(lambda entry: "Global North" if entry["HDI"] >= 0.8 else "Global South", axis = 1)

In [10]:
merged_df["HDI_category"].value_counts()

HDI_category
Global South    3712
Global North    1193
Name: count, dtype: int64

Expectedly, because we set such a high barrier of inclusion to the "Global North" category (HDI >= 0.8), when the global mean for HDI sits around 0.66, there are a lot more "Global South" country entries than "Global North" country entries

### Generating Classification Metric #2 (based on Life Expectancy)

In [11]:
merged_df["LifeExpectancy"].describe()

count    4905.000000
mean       68.940759
std         9.240770
min        37.105000
25%        63.052200
50%        70.981800
75%        75.960600
max        84.425800
Name: LifeExpectancy, dtype: float64

In [12]:
life_expectancy_bins = [0, 70, 79, 85]
life_expectancy_labels = ["Low", "Medium", "High"]

merged_df["LifeExpectancy_category"] = pd.cut(
    merged_df["LifeExpectancy"],
    bins = life_expectancy_bins,
    labels = life_expectancy_labels,
    include_lowest = True,
    right = True
)

In [13]:
merged_df["LifeExpectancy_category"].value_counts()

LifeExpectancy_category
Low       2231
Medium    2094
High       580
Name: count, dtype: int64

Similar to the "Global North"/"Global South" split, we are seeing very few entries in the "High" Life Expectancy bin

### Generating Classification Metric #3 (based on GNI)

In [14]:
merged_df["GNI_per_capita"].describe()

count      4905.000000
mean      17103.661270
std       18567.146877
min         416.032660
25%        3620.024447
50%       10244.273700
75%       24077.246680
max      104640.280700
Name: GNI_per_capita, dtype: float64

We generate our quartiles (Quartile Q1 is the LOWEST, while Quartile Q4 is the HIGHEST)

In [15]:
quartiles = merged_df["GNI_per_capita"].quantile([0, .25, .5, .76, 1]).to_list()
labels = ["Q1", "Q2", "Q3", "Q4"]

quartiles

[416.0326599, 3620.024447, 10244.2737, 25112.9142952, 104640.2807]

In [16]:
merged_df["GNI_quartile"] = pd.cut(merged_df["GNI_per_capita"], bins = quartiles, labels = labels, include_lowest = True, duplicates = "drop")

We check to ensure that each quartile has a comparable amount of countries (so our models don't become biased, or one model "performs better" than the others).

In [17]:
merged_df["GNI_quartile"].value_counts()

GNI_quartile
Q3    1275
Q1    1227
Q2    1226
Q4    1177
Name: count, dtype: int64

In [18]:
# Verifying all our classification metric columns were properly created
merged_df.head()

,Country/Territory,Code,Year,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS,Total Chronic Deaths,GNI_per_capita,HDI,LifeExpectancy,HDI_category,LifeExpectancy_category,GNI_quartile
0,Afghanistan,AFG,1990,1116,371,44899,11580,2108,3709,5945,2673,34,72435,2684.550019,0.273,45.9672,Global South,Low,Q1
1,Afghanistan,AFG,1991,1136,374,45492,11796,2120,3724,6050,2728,41,73461,2276.289409,0.279,46.6631,Global South,Low,Q1
2,Afghanistan,AFG,1992,1162,378,46557,12218,2153,3776,6223,2830,48,75345,2059.868084,0.287,47.5955,Global South,Low,Q1
3,Afghanistan,AFG,1993,1187,384,47951,12634,2195,3862,6445,2943,56,77657,1525.533426,0.297,51.4664,Global South,Low,Q1
4,Afghanistan,AFG,1994,1211,391,49308,12914,2231,3932,6664,3027,63,79741,1087.961890,0.292,51.4945,Global South,Low,Q1


### Calculate Death Ratio

We calculate the death ratio, for use in predicting the "category" of a country.

From now on, the "category" of a country refers to the group it is a part of, when analyzing as per a specific classification metric (see the written report for our classification metric definitions).

In [19]:
chronic_causes = ["Alzheimer's Disease and Other Dementias",
                  "Parkinson's Disease",'Cardiovascular Diseases',
                  'Neoplasms','Diabetes Mellitus','Chronic Kidney Disease',
                  'Chronic Respiratory Diseases','Cirrhosis and Other Chronic Liver Diseases', 'HIV/AIDS']

death_ratio = merged_df[chronic_causes].divide(merged_df['Total Chronic Deaths'], axis = 0)

In [20]:
death_ratio

,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS
0,0.015407,0.005122,0.619852,0.159867,0.029102,0.051205,0.082074,0.036902,0.000469
1,0.015464,0.005091,0.619267,0.160575,0.028859,0.050694,0.082357,0.037135,0.000558
2,0.015422,0.005017,0.617918,0.162161,0.028575,0.050116,0.082593,0.037561,0.000637
3,0.015285,0.004945,0.617472,0.162690,0.028265,0.049732,0.082993,0.037897,0.000721
4,0.015187,0.004903,0.618352,0.161949,0.027978,0.049310,0.083571,0.037960,0.000790
...,...,...,...,...,...,...,...,...,...
5515,0.011099,0.003165,0.245083,0.164297,0.046753,0.031031,0.040496,0.028793,0.429282
5516,0.011500,0.003283,0.253936,0.171894,0.048862,0.032385,0.041800,0.029416,0.406924
5517,0.011994,0.003425,0.263948,0.180358,0.050879,0.033725,0.043277,0.030822,0.381571
5518,0.012594,0.003596,0.276590,0.190698,0.053560,0.035485,0.045132,0.032158,0.350189


# Part 1 of Question 2
How does the death ratio differ across regional classifications?

In [21]:
analyzed_chronic_causes = ["Alzheimer's Disease and Other Dementias", 'HIV/AIDS']

In [22]:
# Statistics on death ratio for "Global South" countries
death_ratio.loc[
    merged_df[merged_df["HDI_category"] == "Global South"].index
][analyzed_chronic_causes].describe()

,Alzheimer's Disease and Other Dementias,HIV/AIDS
count,3712.000000,3712.000000
mean,0.023871,0.102928
std,0.012589,0.177914
min,0.002663,0.000000
25%,0.014353,0.002093
50%,0.022124,0.012693
75%,0.031100,0.116482
max,0.076607,0.780444


In [23]:
# Statistics on death ratio for "Global North" countries
death_ratio.loc[
    merged_df[merged_df["HDI_category"] == "Global North"].index
][analyzed_chronic_causes].describe()

,Alzheimer's Disease and Other Dementias,HIV/AIDS
count,1193.000000,1193.000000
mean,0.049667,0.004361
std,0.019946,0.013576
min,0.005986,0.000041
25%,0.036714,0.000507
50%,0.049020,0.001026
75%,0.061605,0.003442
max,0.145534,0.182381


In [24]:
# Statistics on death ratio for Low Life Expectancy countries
death_ratio.loc[
    merged_df[merged_df["LifeExpectancy_category"] == "Low"].index
][analyzed_chronic_causes].describe()

,Alzheimer's Disease and Other Dementias,HIV/AIDS
count,2231.000000,2231.000000
mean,0.017749,0.162118
std,0.008335,0.208755
min,0.002663,0.000000
25%,0.011549,0.003388
50%,0.016691,0.045012
75%,0.022732,0.269804
max,0.055477,0.780444


In [25]:
# Statistics on death ratio for Medium Life Expectancy countries
death_ratio.loc[
    merged_df[merged_df["LifeExpectancy_category"] == "Medium"].index
][analyzed_chronic_causes].describe()

,Alzheimer's Disease and Other Dementias,HIV/AIDS
count,2094.000000,2094.000000
mean,0.034378,0.011722
std,0.012493,0.021026
min,0.005345,0.000000
25%,0.026197,0.000742
50%,0.033629,0.003404
75%,0.042935,0.013711
max,0.076607,0.187500


In [26]:
# Statistics on death ratio for High Life Expectancy countries
death_ratio.loc[
    merged_df[merged_df["LifeExpectancy_category"] == "High"].index
][analyzed_chronic_causes].describe()

,Alzheimer's Disease and Other Dementias,HIV/AIDS
count,580.000000,580.000000
mean,0.062545,0.001794
std,0.018227,0.002360
min,0.006269,0.000065
25%,0.053530,0.000501
50%,0.061981,0.000750
75%,0.070876,0.002004
max,0.145534,0.013052


In [27]:
# Statistics on death ratio for Q1 GNI countries
death_ratio.loc[
    merged_df[merged_df["GNI_quartile"] == "Q1"].index
][analyzed_chronic_causes].describe()

,Alzheimer's Disease and Other Dementias,HIV/AIDS
count,1227.000000,1227.000000
mean,0.015869,0.208739
std,0.006893,0.218199
min,0.004097,0.000000
25%,0.010553,0.009002
50%,0.015513,0.136345
75%,0.020883,0.359148
max,0.038203,0.780444


In [28]:
# Statistics on death ratio for Q2 GNI countries
death_ratio.loc[
    merged_df[merged_df["GNI_quartile"] == "Q2"].index
][analyzed_chronic_causes].describe()

,Alzheimer's Disease and Other Dementias,HIV/AIDS
count,1226.000000,1226.000000
mean,0.025251,0.059804
std,0.012231,0.135911
min,0.002663,0.000000
25%,0.015425,0.001459
50%,0.023772,0.007281
75%,0.031971,0.031392
max,0.064392,0.726850


In [29]:
# Statistics on death ratio for Q3 GNI countries
death_ratio.loc[
    merged_df[merged_df["GNI_quartile"] == "Q3"].index
][analyzed_chronic_causes].describe()

,Alzheimer's Disease and Other Dementias,HIV/AIDS
count,1275.000000,1275.000000
mean,0.031861,0.038824
std,0.012796,0.108985
min,0.003701,0.000000
25%,0.023035,0.000863
50%,0.030617,0.005304
75%,0.040025,0.019036
max,0.076607,0.718688


In [30]:
# Statistics on death ratio for Q4 GNI countries
death_ratio.loc[
    merged_df[merged_df["GNI_quartile"] == "Q4"].index
][analyzed_chronic_causes].describe()

,Alzheimer's Disease and Other Dementias,HIV/AIDS
count,1177.000000,1177.000000
mean,0.048267,0.007075
std,0.021494,0.030228
min,0.005986,0.000041
25%,0.035514,0.000557
50%,0.048721,0.001175
75%,0.061605,0.004021
max,0.145534,0.472928


# Part 2 of Question 2
Can we predict a country's regional classification with the death ratio?


In [31]:
# For Grid Search
y_hdi_metric = merged_df["HDI_category"]
y_life_metric = merged_df["LifeExpectancy_category"]
y_gni_metric = merged_df["GNI_quartile"]

# first is for static testing, second is for grid search
targets = {
    # Global North / Global South
    # "HDI_category": [X_hdi_training, X_hdi_test, y_hdi_training, y_hdi_test],
    "HDI_category": y_hdi_metric,
    # Life Expectancy Categories
    # "LifeExpectancy_category": [X_le_training, X_le_test, y_le_training, y_le_test],
    "LifeExpectancy_category": y_life_metric,
    # GNI Quartiles
    # "GNI_quartile": [X_gni_training, X_gni_test, y_gni_training, y_gni_test],
    "GNI_quartile": y_gni_metric
}

param_grid = {
    "standardscaler__with_mean": [True, False],
    "standardscaler__with_std": [True, False],
    "kneighborsclassifier__n_neighbors": range(1, 500, 5),
    "kneighborsclassifier__metric": ["euclidean", "manhattan", "minkowski"]
}

In [32]:
merged_df.head()

,Country/Territory,Code,Year,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS,Total Chronic Deaths,GNI_per_capita,HDI,LifeExpectancy,HDI_category,LifeExpectancy_category,GNI_quartile
0,Afghanistan,AFG,1990,1116,371,44899,11580,2108,3709,5945,2673,34,72435,2684.550019,0.273,45.9672,Global South,Low,Q1
1,Afghanistan,AFG,1991,1136,374,45492,11796,2120,3724,6050,2728,41,73461,2276.289409,0.279,46.6631,Global South,Low,Q1
2,Afghanistan,AFG,1992,1162,378,46557,12218,2153,3776,6223,2830,48,75345,2059.868084,0.287,47.5955,Global South,Low,Q1
3,Afghanistan,AFG,1993,1187,384,47951,12634,2195,3862,6445,2943,56,77657,1525.533426,0.297,51.4664,Global South,Low,Q1
4,Afghanistan,AFG,1994,1211,391,49308,12914,2231,3932,6664,3027,63,79741,1087.961890,0.292,51.4945,Global South,Low,Q1


**THIS IS ONE OF THE MOST COMPUTATIONALLY INTENSIVE CELLS IN THE NOTEBOOK (upwards of 10 mins to run on a Jupyter server)** -- take care when running

In [ ]:
%%time

results = {}
confusion_matrices = {}

for target_name, y in targets.items():
    # X_training, X_test, y_training, y_test = y[0], y[1], y[2], y[3]

    # Semantic assignments so that I can test train-test split and grid search
    # side-by-side
    X_training = death_ratio
    X_test = death_ratio
    y_training = y
    y_test = y

    pipeline_knn = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier()
    )

    # Grid search -- MOST COMPUTATIONALLY INTENSIVE PART OF THE NOTEBOOK
    grid_search = GridSearchCV(
        estimator = pipeline_knn,
        param_grid = param_grid,
        cv = 5,
        scoring = "accuracy",
        n_jobs = -1
    )

    grid_search.fit(X_training, y_training)

    y_pred = grid_search.predict(X_test)

    accuracy_knn = accuracy_score(y_test, y_pred)

    # Find precision/recall/f1 scores per each classification group (`target_name`)
    precisions = {}
    recalls = {}
    f1_scores = {}

    for category in list(merged_df[target_name].unique()):
        precisions["Precision_"+category] = precision_score(y_test == category, y_pred == category)
        recalls["Recall_"+category] = recall_score(y_test == category, y_pred == category)
        f1_scores["F1_"+category] = f1_score(y_test == category, y_pred == category)

    target_info = {
        ("Accuracy_"+target_name): accuracy_knn,
        "Best n-value": grid_search.best_params_["kneighborsclassifier__n_neighbors"],
        "Best metric": grid_search.best_params_["kneighborsclassifier__metric"],
        "with_mean": grid_search.best_params_["standardscaler__with_mean"],
        "with_std": grid_search.best_params_["standardscaler__with_std"]
    }
    target_info.update(precisions)
    target_info.update(recalls)
    target_info.update(f1_scores)

    results[target_name] = {
        "scoring_metrics": pd.Series(
            target_info,
            name = target_name
        ),
        "confusion_matrix": pd.DataFrame(
            confusion_matrix(y_test, y_pred),
            index = np.unique(merged_df[target_name]),
            columns = np.unique(merged_df[target_name])
        )
    }

In [35]:
results["HDI_category"]["scoring_metrics"]

Accuracy_HDI_category      0.923344
Best n-value                     61
Best metric               euclidean
with_mean                      True
with_std                       True
Precision_Global South     0.936421
Precision_Global North     0.877193
Recall_Global South         0.96417
Recall_Global North        0.796312
F1_Global South            0.950093
F1_Global North            0.834798
Name: HDI_category, dtype: object

In [25]:
results["HDI_category"]["confusion_matrix"]

,Global North,Global South
Global North,950,243
Global South,133,3579


In [26]:
results["LifeExpectancy_category"]["scoring_metrics"]

Accuracy_LifeExpectancy_category     0.856065
Best n-value                              276
Best metric                         manhattan
with_mean                                True
with_std                                 True
Precision_Low                         0.88681
Precision_Medium                     0.808994
Precision_High                       0.938034
Recall_Low                            0.87091
Recall_Medium                        0.867717
Recall_High                          0.756897
F1_Low                               0.878788
F1_Medium                            0.837327
F1_High                              0.837786
Name: LifeExpectancy_category, dtype: object

In [28]:
results["LifeExpectancy_category"]["confusion_matrix"]

,High,Low,Medium
High,439,0,141
Low,0,1943,288
Medium,29,248,1817


In [29]:
results["GNI_quartile"]["scoring_metrics"]

Accuracy_GNI_quartile     0.721509
Best n-value                    86
Best metric              euclidean
with_mean                     True
with_std                      True
Precision_Q1              0.719921
Precision_Q2              0.649886
Precision_Q3              0.667865
Precision_Q4              0.846291
Recall_Q1                 0.892421
Recall_Q2                 0.463295
Recall_Q3                 0.728627
Recall_Q4                 0.804588
F1_Q1                     0.796943
F1_Q2                     0.540952
F1_Q3                     0.696924
F1_Q4                     0.824913
Name: GNI_quartile, dtype: object

In [30]:
results["GNI_quartile"]["confusion_matrix"]

,Q1,Q2,Q3,Q4
Q1,1095,97,29,6
Q2,330,568,271,57
Q3,77,160,929,109
Q4,19,49,162,947


From these scoring metrics and confusion matrices, we can deduce a few conclusions. First, we see that we are getting a relatively high success rate for predicting "Global North"/"Global South" countries (accuracy ~ 92.33%), as well as predicting the Life Expectancy category of a country (accuracy ~ 85.60%). However, we do note that the accuracy of predicting the GNI quartile of a country is comparably very poor (accuracy ~ 72.15%). We analyze this model, along with Part 1, further in the written report.